### HOMEWORK 2 – Data Wrangling with pandas

Raul Miranda

Feb 26, 2026

---

#### Learning Objectives
By completing this assignment, you should be able to:

- Use pandas equivalents of `dplyr` verbs:
  - `filter()` → boolean indexing / `.loc[]`
  - `mutate()` → `.assign()`
  - `group_by()` → `.groupby()`
  - `summarise()` → `.agg()`
  - `arrange()` → `.sort_values()`
- Apply boolean logic correctly using `&` and `|`
- Use **method chaining** for readable, step-by-step transformations
- Create grouped summary tables with multiple statistics

---

In [2]:
### Dataset: provided housing.csv

#### Here I'm reading from gitbhub; alternatively from my working directory

import pandas as pd

#df = pd.read_csv("housing.csv") ; if reading from my working directory

df = pd.read_csv("https://raw.githubusercontent.com/raul-miranda/DS201-2026/refs/heads/main/Week%202/housing.csv")


### PART A - Core Wrangling (Method Chaining Required) **

#### Some quick info and stats to begin:

In [3]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   listing_id    600 non-null    int64  
 1   price         600 non-null    int64  
 2   size          547 non-null    float64
 3   bedrooms      576 non-null    float64
 4   neighborhood  600 non-null    object 
 5   type          600 non-null    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 28.3+ KB


,listing_id,price,size,bedrooms
count,600.000000,600.0,547.000000,576.000000
mean,100300.500000,1500000.0,1763.208584,2.244792
std,173.349358,0.0,701.737347,1.308490
min,100001.000000,1500000.0,607.197941,0.000000
25%,100150.750000,1500000.0,1285.926707,1.000000
50%,100300.500000,1500000.0,1626.257658,2.000000
75%,100450.250000,1500000.0,2081.737227,3.000000
max,100600.000000,1500000.0,4500.000000,6.000000


#### Here we're filtering houses priced above 250K, sized above 1K, defining new variable price_per_sqft, grouping by neighborhood,aggregating rows by neighborhood and calculating mean, median and count of houses, and finally sorting in descending mean_price_sqft

#### Everything is chained: query, assign, groupby, agg, and then sort_values

#### The table called summary is the default python table with neighborhood as index and mean, median and count as columns

In [4]:

summary = (
    df
    .loc[(df['price'] > 250000) & (df['size'] > 1000)]
    .assign(price_per_sqft = lambda x: x['price'] / x['size'])
    .groupby('neighborhood')
    .agg(
        mean_price_sqft=('price_per_sqft', 'mean'),
        median_price_sqft=('price_per_sqft', 'median'),
        home_count=('price_per_sqft', 'count')
    )
    .sort_values('mean_price_sqft', ascending=False)
)
summary

,mean_price_sqft,median_price_sqft,home_count
neighborhood,,,
Downtown,977.820905,1001.557049,99
Midtown,921.141446,901.992377,92
Suburb,861.917078,836.878644,157
Uptown,860.935608,843.653468,99
Waterfront,849.508891,792.663291,48



### Part B – Translation to dplyr

#### Here's the r equivalent code to do the same thing

summary <- df |>
  filter(price > 250000, size > 1000) |>
  mutate(price_per_sqft = price / size) |>
  group_by(neighborhood) |>
  summarise(
    mean_price_sqft = mean(price_per_sqft),
    median_price_sqft = median(price_per_sqft),
    home_count = n()
  ) |>
  arrange(desc(mean_price_sqft))

#### Reflection

To me, both chaining or piping syntaxes are equivalent once one gets used to them.  But since I learned R before Python,the dplyr piping seems cleaner because |> clearly shows how ouputs from one command go into the next commands. Now, the commands in R and Python perform identical actions,  but dplyr is more direct. Clearly 'filter' is more obvious than using the accessor .loc. Also, dplyr allows column names without quotes (pandas requires quotes or lambda functions). Finally, the summarize and agg are almost similar in form, but summarize() seems more direct, while agg() requires a special syntax to get the summary stats calculated. 


### Part C – Boolean Logic Debugging

In [5]:
df[df["price"] > 250000 & df["size"] > 1000]

TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]

#### Error explanation: the error type indicates that the command is trying to do boolean & between a float (25000) and a boolean series, which are incompatible. 

#### The fix is to separately evaluate the left boolean and the right boolean and then compare the two series. To do so, one can enclose the booleans between parentheses.

In [ ]:
df[(df["price"] > 250000) & (df["size"] > 1000)]

,listing_id,price,size,bedrooms,neighborhood,type
0,100001,1500000,1280.741760,1.0,Suburb,Townhouse
1,100002,1500000,1406.283113,2.0,Uptown,SingleFamily
2,100003,1500000,4146.825713,6.0,Suburb,MultiFamily
3,100004,1500000,3946.599818,6.0,Suburb,SingleFamily
4,100005,1500000,1243.751760,1.0,Downtown,MultiFamily
...,...,...,...,...,...,...
595,100596,1500000,1443.241197,3.0,Midtown,Condo
596,100597,1500000,1083.909714,2.0,Suburb,Condo
597,100598,1500000,1600.126432,1.0,Suburb,SingleFamily
598,100599,1500000,1248.216637,1.0,Waterfront,Condo


#### Rewriting the filter with the .query method, which is much cleaner and allows both filters within the single subsetting of df. 

In [ ]:
df.query("price > 250000 and size > 1000")

,listing_id,price,size,bedrooms,neighborhood,type
0,100001,1500000,1280.741760,1.0,Suburb,Townhouse
1,100002,1500000,1406.283113,2.0,Uptown,SingleFamily
2,100003,1500000,4146.825713,6.0,Suburb,MultiFamily
3,100004,1500000,3946.599818,6.0,Suburb,SingleFamily
4,100005,1500000,1243.751760,1.0,Downtown,MultiFamily
...,...,...,...,...,...,...
595,100596,1500000,1443.241197,3.0,Midtown,Condo
596,100597,1500000,1083.909714,2.0,Suburb,Condo
597,100598,1500000,1600.126432,1.0,Suburb,SingleFamily
598,100599,1500000,1248.216637,1.0,Waterfront,Condo


## Part D – Short Concept Questions

1. Why must we wrap each condition in parentheses when using `&` in pandas?


Because & has higher precedence then > or <  operators, and thus is evaluated first. The parentheses ensure that the > and < comparisons are evaluated before the &.


2. What is the advantage of method chaining over creating many temporary DataFrames?

TO me the clear advantage is clarity and compactness of the code. It also helps avoid errors by referring to the wrong subset dataframe later in the code.


3. In `.agg(mean_price=("price", "mean"))`, what does `"price"` represent? What does `"mean"` represent?

"price" is the column name containing individual prices in each row.  While "mean" is the function applied to the "price" column values.

4. When you `groupby("neighborhood")`, why does `neighborhood` appear on the left (index) in the result table?

By design Pandas uses the groupby column as the index and identifier of each of the rows in the the summary table. It moves that identifier ("neighborhood") to the left and is no longer a column.

### Optional extension of a summary table by type reporting mean and median price and count of homes.  then I'll sort by mean price.

In [6]:
summary2 = (
    df
    .groupby('type')
    .agg(
        mean_price=('price', 'mean'),
        median_price=('price', 'median'),
        homes_count=('price', 'count')
    )
    .sort_values('mean_price', ascending=False)
)
summary2

,mean_price,median_price,homes_count
type,,,
Condo,1500000.0,1500000.0,183
MultiFamily,1500000.0,1500000.0,63
SingleFamily,1500000.0,1500000.0,235
Townhouse,1500000.0,1500000.0,119


#### comment: the original dataset contains only one home price for all homes, hence that result.